In [60]:
from pathlib import Path
import json
import pandas as pd

try:
    from IPython.display import display  # type: ignore
except Exception:
    # Fallback for non-notebook execution
    display = print

# Reload local notebook utilities to pick up edits without restarting kernel
import importlib
import testLibs as tl
importlib.reload(tl)

# Notebook helpers (scoring + flatteners)
ResultsFlattener = tl.ResultsFlattener
mznResultsFlattener = tl.mznResultsFlattener
get_significative_solvers = tl.get_significative_solvers
scoreComputation_subset = tl.scoreComputation_subset
compute_llm_scores = tl.compute_llm_scores
compute_top1_llm_scores = tl.compute_top1_llm_scores
compute_closed_gap = tl.compute_closed_gap
build_llm_performance_table = tl.build_llm_performance_table
filter_to_solvers = tl.filter_to_solvers
singleSolverScore = tl.singleSolverScore
scoreComputation = tl.scoreComputation

# Optional plotting libs (only needed for plots)
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None
try:
    import seaborn as sns
except ModuleNotFoundError:
    sns = None

In [61]:
data_dir = Path("../data/testOutputSwappedFree").resolve()
if not data_dir.is_dir():
    raise FileNotFoundError(f"Expected folder not found: {data_dir}")

json_paths = sorted(data_dir.glob("*.json"))
print(f"Loading {len(json_paths)} JSON files from {data_dir}")

data_by_file = {}
for p in json_paths:
    with p.open("r", encoding="utf-8") as f:
        data_by_file[p.name] = json.load(f)

# List in a stable order, matching json_paths
data_list = [data_by_file[p.name] for p in json_paths]

print("Loaded files:")
for name in data_by_file.keys():
    print(" -", name)

with open('../data/tablesJSON/allTables_free.json', 'r') as f1:
    MznResults = json.load(f1)

Loading 5 JSON files from /home/vro5/Coding/AgenticSolvers/test/data/testOutputSwappedFree
Loaded files:
 - LLMsuggestions_swapped_chat_fzn_Sdesc_T0p2.json
 - LLMsuggestions_swapped_chat_fzn_Sdesc_T0p7.json
 - LLMsuggestions_swapped_featOnly_Sdesc_T0p3.json
 - LLMsuggestions_swapped_featOnly_Sdesc_T0p7.json
 - LLMsuggestions_swapped_uncommented_desc_solverdesc.json


In [62]:
# --- Full free-solvers analysis (parallel, single, closed-gap) ---
# Flatten MiniZinc results and compute scores over the full free-solver set
mzn_raw_df = mznResultsFlattener(MznResults)
scored_df = scoreComputation(mzn_raw_df)
print(f"Computed solver scores: {len(scored_df)} rows")

# --- Top-3 (parallel) score evaluation per input file ---
score_parts = []
for fname, llm_results in data_by_file.items():
    llm_df = ResultsFlattener(llm_results)
    if llm_df is None or llm_df.empty:
        continue
    if 'top3_list' not in llm_df.columns:
        # skip if no parallel suggestions
        continue
    needed = ['model', 'problem', 'instance', 'top3_list']
    llm_df_clean = llm_df[[c for c in needed if c in llm_df.columns]].copy()
    llm_df_clean['provider'] = 'all'
    part = compute_llm_scores(llm_df_clean, scored_df)
    part = part.drop(columns=['provider'], errors='ignore')
    part['source_file'] = fname
    score_parts.append(part)

llm_top3_by_file = pd.concat(score_parts, ignore_index=True) if score_parts else pd.DataFrame()
if not llm_top3_by_file.empty:
    llm_top3_by_file = llm_top3_by_file.sort_values(['LLM_TotalScore', 'LLM_AvgScore'], ascending=[False, False])
    display(llm_top3_by_file)
else:
    display("No Top-3 (parallel) scores produced")

# --- Top-1 score evaluation per input file ---
top1_score_parts = []
llm_top1_scored_by_file = {}
for fname, llm_results in data_by_file.items():
    llm_df = ResultsFlattener(llm_results)
    if llm_df is None or llm_df.empty:
        continue
    if 'top1' not in llm_df.columns:
        print(f"Skipping Top-1 for {fname}: missing 'top1' column")
        continue
    needed = ['model', 'problem', 'instance', 'top1']
    llm_df_clean = llm_df[[c for c in needed if c in llm_df.columns]].copy()
    llm_df_clean['provider'] = 'all'
    top1_summary, top1_scored = compute_top1_llm_scores(llm_df_clean, scored_df)
    top1_summary = top1_summary.drop(columns=['provider'], errors='ignore')
    top1_summary['source_file'] = fname
    top1_score_parts.append(top1_summary)
    llm_top1_scored_by_file[fname] = top1_scored

llm_top1_by_file = pd.concat(top1_score_parts, ignore_index=True) if top1_score_parts else pd.DataFrame()
if not llm_top1_by_file.empty:
    llm_top1_by_file = llm_top1_by_file.sort_values(['LLM_Top1_TotalScore', 'LLM_Top1_AvgScore'], ascending=[False, False])
    display(llm_top1_by_file)
else:
    display("No Top-1 scores produced")

# --- Closed Gap per input file ---
cg_parts = []
for fname, top1_scored in llm_top1_scored_by_file.items():
    if top1_scored is None or (isinstance(top1_scored, pd.DataFrame) and top1_scored.empty):
        continue
    cg_results = compute_closed_gap(top1_scored, scored_df)
    if not cg_results:
        continue
    part = pd.DataFrame(cg_results)
    part = part.drop(columns=['provider'], errors='ignore')
    part['source_file'] = fname
    cg_parts.append(part)

closed_gap_by_file = pd.concat(cg_parts, ignore_index=True) if cg_parts else pd.DataFrame()
if not closed_gap_by_file.empty:
    closed_gap_by_file = closed_gap_by_file.sort_values(['source_file', 'ClosedGap'], ascending=[True, False]).reset_index(drop=True)
    display(closed_gap_by_file)
else:
    display("No Closed Gap results produced")

# --- Summary table: per input file best Single, Parallel, ClosedGap ---
def _best_by_file(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    if df is None or df.empty or value_col not in df.columns:
        return pd.DataFrame(columns=['source_file', value_col])
    out = df.dropna(subset=['source_file']).copy()
    out[value_col] = pd.to_numeric(out[value_col], errors='coerce')
    out = out.dropna(subset=[value_col])
    if out.empty:
        return pd.DataFrame(columns=['source_file', value_col])
    out = (
        out.sort_values(['source_file', value_col], ascending=[True, False])
        .groupby(['source_file'], as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
    return out[['source_file', value_col]].copy()

single_best = _best_by_file(llm_top1_by_file, 'LLM_Top1_TotalScore')
parallel_best = _best_by_file(llm_top3_by_file, 'LLM_TotalScore')
cg_best = _best_by_file(closed_gap_by_file, 'ClosedGap')

summary = single_best.merge(parallel_best, on='source_file', how='outer')
summary = summary.merge(cg_best, on='source_file', how='outer')
if summary.empty:
    display("No summary rows available yet — run the cells above.")
else:
    summary = summary.rename(columns={
        'source_file': 'File',
        'LLM_Top1_TotalScore': 'Single Score',
        'LLM_TotalScore': 'Parallel Score',
        'ClosedGap': 'Closed Gap'
    })
    summary = summary[['File', 'Single Score', 'Parallel Score', 'Closed Gap']]
    summary['Single Score'] = pd.to_numeric(summary['Single Score'], errors='coerce')
    summary['Parallel Score'] = pd.to_numeric(summary['Parallel Score'], errors='coerce')
    summary['Closed Gap'] = pd.to_numeric(summary['Closed Gap'], errors='coerce')
    summary = summary.sort_values(['Single Score', 'Parallel Score'], ascending=[False, False], na_position='last').reset_index(drop=True)
    display(summary)


Computed solver scores: 2000 rows


,model,LLM_TotalScore,InstancesCovered,LLM_AvgScore,source_file
3,openai/gpt-oss-120b,83.025694,100,0.830257,LLMsuggestions_swapped_featOnly_Sdesc_T0p7.json
2,openai/gpt-oss-120b,82.903225,100,0.829032,LLMsuggestions_swapped_featOnly_Sdesc_T0p3.json
4,openai/gpt-oss-120b,81.491962,100,0.814920,LLMsuggestions_swapped_uncommented_desc_solver...
1,openai/gpt-oss-120b,81.311529,100,0.813115,LLMsuggestions_swapped_chat_fzn_Sdesc_T0p7.json
0,openai/gpt-oss-120b,79.880995,100,0.798810,LLMsuggestions_swapped_chat_fzn_Sdesc_T0p2.json


,model,LLM_Top1_TotalScore,LLM_Top1_AvgScore,InstancesCovered,source_file
2,openai/gpt-oss-120b,77.583085,0.775831,100,LLMsuggestions_swapped_featOnly_Sdesc_T0p3.json
3,openai/gpt-oss-120b,75.243533,0.752435,100,LLMsuggestions_swapped_featOnly_Sdesc_T0p7.json
4,openai/gpt-oss-120b,74.172913,0.741729,100,LLMsuggestions_swapped_uncommented_desc_solver...
0,openai/gpt-oss-120b,59.780855,0.597809,100,LLMsuggestions_swapped_chat_fzn_Sdesc_T0p2.json
1,openai/gpt-oss-120b,52.510323,0.525103,100,LLMsuggestions_swapped_chat_fzn_Sdesc_T0p7.json


,model,InstancesCovered,AS,SBS,VBS,ClosedGap,source_file
0,openai/gpt-oss-120b,100,59.780855,76.964375,89.0,-1.427721,LLMsuggestions_swapped_chat_fzn_Sdesc_T0p2.json
1,openai/gpt-oss-120b,100,52.510323,76.964375,89.0,-2.031806,LLMsuggestions_swapped_chat_fzn_Sdesc_T0p7.json
2,openai/gpt-oss-120b,100,77.583085,76.964375,89.0,0.051407,LLMsuggestions_swapped_featOnly_Sdesc_T0p3.json
3,openai/gpt-oss-120b,100,75.243533,76.964375,89.0,-0.142979,LLMsuggestions_swapped_featOnly_Sdesc_T0p7.json
4,openai/gpt-oss-120b,100,74.172913,76.964375,89.0,-0.231933,LLMsuggestions_swapped_uncommented_desc_solver...


,File,Single Score,Parallel Score,Closed Gap
0,LLMsuggestions_swapped_featOnly_Sdesc_T0p3.json,77.583085,82.903225,0.051407
1,LLMsuggestions_swapped_featOnly_Sdesc_T0p7.json,75.243533,83.025694,-0.142979
2,LLMsuggestions_swapped_uncommented_desc_solver...,74.172913,81.491962,-0.231933
3,LLMsuggestions_swapped_chat_fzn_Sdesc_T0p2.json,59.780855,79.880995,-1.427721
4,LLMsuggestions_swapped_chat_fzn_Sdesc_T0p7.json,52.510323,81.311529,-2.031806


In [63]:
# --- Reconduction: remap suggested solver names to the original descriptions and re-score ---
from pathlib import Path
import json
import re

# Load swap mapping
swap_path = Path("../data/swappedFreeSolversDesc.json").resolve()
if not swap_path.is_file():
    raise FileNotFoundError(f"Expected mapping file not found: {swap_path}")
with swap_path.open("r", encoding="utf-8") as sf:
    swap_data = json.load(sf)

remap = {s["original"]: s["description_from"] for s in swap_data.get("swaps", [])}
print(f"Loaded remap for {len(remap)} solvers")

# Helper to remap top3/top1 entries
def _remap_top3_list(val):
    if val is None:
        return None
    if isinstance(val, list):
        return [remap.get(str(v).strip(), str(v).strip()) for v in val]
    # handle string encoded lists
    parts = [p.strip() for p in str(val).replace(';', ',').split(',') if p.strip()]
    return [remap.get(p, p) for p in parts] if parts else None

def _remap_top1(val):
    if val is None:
        return None
    s = str(val).strip()
    return remap.get(s, s)

# Re-score each input file after remapping suggestions
recon_top3_parts = []
recon_top1_parts = []
recon_cg_parts = []
recon_tables_by_file = {}

for fname, llm_results in data_by_file.items():
    llm_df = ResultsFlattener(llm_results)
    if llm_df is None or llm_df.empty:
        continue

    # create remapped copy
    df2 = llm_df.copy()
    if 'top3_list' in df2.columns:
        df2['top3_list'] = df2['top3_list'].apply(_remap_top3_list)
    if 'top1' in df2.columns:
        df2['top1'] = df2['top1'].apply(_remap_top1)

    # compute scores using full scored_df (free solvers)
    top3_summary = compute_llm_scores(df2, scored_df)
    top1_summary, top1_scored = compute_top1_llm_scores(df2, scored_df)
    cg_rows = compute_closed_gap(top1_scored, scored_df, allowed_solvers=None, sbs_solver=None)
    cg_df = pd.DataFrame(cg_rows)
    if 'ClosedGap' not in cg_df.columns:
        for c in ['provider', 'model', 'InstancesCovered', 'AS', 'SBS', 'VBS', 'ClosedGap']:
            if c not in cg_df.columns:
                cg_df[c] = pd.NA

    perf_table = build_llm_performance_table(
        top3_summary=top3_summary,
        top1_summary=top1_summary,
        closed_gap=cg_df,
        sort_by='SingleScore',
        ascending=False,
    )

    recon_tables_by_file[fname] = {
        'top3_summary': top3_summary,
        'top1_summary': top1_summary,
        'closed_gap': cg_df,
        'performance_table': perf_table,
    }

    # Collect parts for aggregated views
    if top3_summary is not None and not top3_summary.empty:
        t3 = top3_summary.drop(columns=['provider'], errors='ignore')
        t3['source_file'] = fname
        recon_top3_parts.append(t3)
    if top1_summary is not None and not top1_summary.empty:
        t1 = top1_summary.drop(columns=['provider'], errors='ignore')
        t1['source_file'] = fname
        recon_top1_parts.append(t1)
    if not cg_df.empty:
        cg_part = cg_df.drop(columns=['provider'], errors='ignore')
        cg_part['source_file'] = fname
        recon_cg_parts.append(cg_part)

print(f"Re-scored {len(recon_tables_by_file)} files after remapping")

recon_top3_all = pd.concat(recon_top3_parts, ignore_index=True) if recon_top3_parts else pd.DataFrame()
recon_top1_all = pd.concat(recon_top1_parts, ignore_index=True) if recon_top1_parts else pd.DataFrame()
recon_cg_all = pd.concat(recon_cg_parts, ignore_index=True) if recon_cg_parts else pd.DataFrame()

# Display per-file performance tables (stable order)
for p in json_paths:
    fname = p.name
    if fname not in recon_tables_by_file:
        continue
    print("\n" + "=" * 80)
    print("Reconduced File:", fname)
    print("=" * 80)
    print("\nPerformance table:")
    display(recon_tables_by_file[fname]['performance_table'])

# Aggregated summary per file: best Single, Parallel, ClosedGap after reconduction
def _best_by_file(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    if df is None or df.empty or value_col not in df.columns:
        return pd.DataFrame(columns=['source_file', value_col])
    out = df.dropna(subset=['source_file']).copy()
    out[value_col] = pd.to_numeric(out[value_col], errors='coerce')
    out = out.dropna(subset=[value_col])
    if out.empty:
        return pd.DataFrame(columns=['source_file', value_col])
    out = (
        out.sort_values(['source_file', value_col], ascending=[True, False])
        .groupby(['source_file'], as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
    return out[['source_file', value_col]].copy()

single_best = _best_by_file(recon_top1_all, 'LLM_Top1_TotalScore')
parallel_best = _best_by_file(recon_top3_all, 'LLM_TotalScore')
cg_best = _best_by_file(recon_cg_all, 'ClosedGap')

summary_recon = single_best.merge(parallel_best, on='source_file', how='outer')
summary_recon = summary_recon.merge(cg_best, on='source_file', how='outer')
if summary_recon.empty:
    display("No reconduced summary rows available")
else:
    summary_recon = summary_recon.rename(columns={
        'source_file': 'File',
        'LLM_Top1_TotalScore': 'Single Score',
        'LLM_TotalScore': 'Parallel Score',
        'ClosedGap': 'Closed Gap'
    })
    summary_recon = summary_recon[['File', 'Single Score', 'Parallel Score', 'Closed Gap']]
    summary_recon['Single Score'] = pd.to_numeric(summary_recon['Single Score'], errors='coerce')
    summary_recon['Parallel Score'] = pd.to_numeric(summary_recon['Parallel Score'], errors='coerce')
    summary_recon['Closed Gap'] = pd.to_numeric(summary_recon['Closed Gap'], errors='coerce')

    # Map filenames -> Variant labels (like fzn2nlAnalysis)
    def _map_variant_from_filename(fname: str) -> str:
        if fname is None:
            return ''
        fn = str(fname)
        if 'featOnly_Sdesc' in fn or 'featOnly_Sdesc' in fn or 'featOnly_Sdesc' in fn:
            return 'Features + Solvers Description'
        if 'featOnly_Pdesc_Sdesc' in fn or 'featOnly_Pdesc' in fn:
            return 'Features + Problem Description'
        if 'uncommented_desc' in fn or 'uncommented' in fn or 'uncommented_desc_solverdesc' in fn:
            return 'Scripts + Problem Description + Solvers Description'
        if 'fzn_Sdesc' in fn or 'chat_fzn_Sdesc' in fn or fn.startswith('LLMsuggestions_chat_fzn'):
            return 'fzn2nl + Solver Description'
        # default: keep filename (but user requested removing File column later)
        return fn

    def _extract_temp_from_filename(fname: str) -> str:
        if fname is None:
            return ''
        m = re.search(r"(T\d+(?:p\d+)?)", str(fname))
        return m.group(1) if m else ''

    summary_recon['Variant'] = summary_recon['File'].apply(_map_variant_from_filename)
    summary_recon['Temperature'] = summary_recon['File'].apply(_extract_temp_from_filename)

    # Reorder to include Variant and Temperature, drop File
    summary_recon = summary_recon[['Variant', 'Temperature', 'Single Score', 'Parallel Score', 'Closed Gap']]

    summary_recon = summary_recon.sort_values(['Single Score', 'Parallel Score'], ascending=[False, False], na_position='last').reset_index(drop=True)
    display(summary_recon.style.hide(axis="index"))


Loaded remap for 20 solvers
Re-scored 5 files after remapping

Reconduced File: LLMsuggestions_swapped_chat_fzn_Sdesc_T0p2.json

Performance table:


,Model,Single Score,Parallel Score,Closed Gap
0,openai/gpt-oss-120b,58.827415,80.53447,-1.50694



Reconduced File: LLMsuggestions_swapped_chat_fzn_Sdesc_T0p7.json

Performance table:


,Model,Single Score,Parallel Score,Closed Gap
0,openai/gpt-oss-120b,60.509869,81.068252,-1.36715



Reconduced File: LLMsuggestions_swapped_featOnly_Sdesc_T0p3.json

Performance table:


...


Reconduced File: LLMsuggestions_swapped_featOnly_Sdesc_T0p7.json

Performance table:


,Model,Single Score,Parallel Score,Closed Gap
0,openai/gpt-oss-120b,50.797299,77.640982,-2.174135



Reconduced File: LLMsuggestions_swapped_uncommented_desc_solverdesc.json

Performance table:


,Model,Single Score,Parallel Score,Closed Gap
0,openai/gpt-oss-120b,46.154496,70.744078,-2.55989


Variant,Temperature,Single Score,Parallel Score,Closed Gap
fzn2nl + Solver Description,T0p7,60.509869,81.068252,-1.367150
fzn2nl + Solver Description,T0p2,58.827415,80.534470,-1.506940
Features + Solvers Description,T0p7,50.797299,77.640982,-2.174135
Features + Solvers Description,T0p3,47.712594,73.080124,-2.430433
Scripts + Problem Description + Solvers Description,,46.154496,70.744078,-2.559890


In [ ]:
# --- Suggestion occurrence analysis per file, grouped per Variant+Temperature ---
from collections import Counter, defaultdict
import re

# Build inverse remap: description_from -> original solver
inv_remap = {v: k for k, v in remap.items()} if 'remap' in globals() else {}
# If swap_data is available, prefer constructing from it
if 'swap_data' in globals() and isinstance(swap_data, dict):
    inv_from_swap = {s.get('description_from'): s.get('original') for s in swap_data.get('swaps', [])}
    inv_remap.update({k: v for k, v in inv_from_swap.items() if k is not None})

# Helper: same variant mapping used earlier
def _map_variant_from_filename(fname: str) -> str:
    if fname is None:
        return ''
    fn = str(fname)
    if 'featOnly_Sdesc' in fn or 'featOnly_Sdes' in fn or 'featOnly_Sdesc' in fn:
        return 'Features + Solvers Description'
    if 'featOnly_Pdesc_Sdesc' in fn or 'featOnly_Pdesc' in fn:
        return 'Features + Problem Description'
    if 'uncommented_desc' in fn or 'uncommented' in fn or 'uncommented_desc_solverdesc' in fn:
        return 'Scripts + Problem Description + Solvers Description'
    if 'fzn_Sdesc' in fn or 'chat_fzn_Sdesc' in fn or fn.startswith('LLMsuggestions_chat_fzn'):
        return 'fzn2nl + Solver Description'
    return 'Other'

# Helper: extract temperature token from filename (e.g., T0p7, T0p8, T0)
def _extract_temp_from_filename(fname: str) -> str:
    if fname is None:
        return ''
    m = re.search(r"(T\d+(?:p\d+)?)", fname)
    return m.group(1) if m else ''

occurrence_tables_by_group = defaultdict(list)

for p in json_paths:
    fname = p.name
    llm_results = data_by_file.get(fname)
    if llm_results is None:
        continue
    df = ResultsFlattener(llm_results)
    if df is None or df.empty:
        continue

    top1_counter = Counter()
    total_counter = Counter()

    # Count occurrences per instance but ensure TotalCount counts unique instances
    for _, row in df.iterrows():
        instance_seen = set()
        # top1
        if 'top1' in df.columns:
            v = row.get('top1')
            if v is not None and pd.notna(v):
                s = str(v).strip()
                if s:
                    top1_counter[s] += 1
                    instance_seen.add(s)
        # top3_list (parallel suggestions)
        if 'top3_list' in df.columns:
            lst = row.get('top3_list')
            if isinstance(lst, list):
                for it in lst:
                    if it is None:
                        continue
                    s = str(it).strip()
                    if s:
                        instance_seen.add(s)
            else:
                parts = [pp.strip() for pp in str(lst).replace(';', ',').split(',') if pp.strip()]
                for it in parts:
                    instance_seen.add(it)
        # increment total_counter once per solver for this instance
        for s in instance_seen:
            total_counter[s] += 1

    # Build per-file table rows
    names = sorted(set(list(total_counter.keys()) + list(top1_counter.keys())))
    occ_rows = []
    for name in names:
        orig = inv_remap.get(name, pd.NA)
        occ_rows.append({
            'SuggestedName': name,
            'OriginalSolver': orig,
            'Top1Count': int(top1_counter.get(name, 0)),
            'TotalCount': int(total_counter.get(name, 0)),
            'File': fname,
        })
    if not occ_rows:
        continue
    occ_df = pd.DataFrame(occ_rows)
    occ_df['Variant'] = occ_df['File'].apply(_map_variant_from_filename)
    occ_df['Temperature'] = occ_df['File'].apply(_extract_temp_from_filename)

    key = (occ_df.iloc[0]['Variant'], occ_df.iloc[0]['Temperature'])
    occurrence_tables_by_group[key].append(occ_df)

# Display one table per Variant+Temperature (concatenate per-files within group)
for (variant, temp), tables in sorted(occurrence_tables_by_group.items()):
    header = f"Suggestion counts — Variant: {variant} / Temp: {temp if temp else 'unknown'}"
    print('\n' + '=' * 80)
    print(header)
    print('=' * 80)
    group_df = pd.concat(tables, ignore_index=True)
    # Show per-file counts, sorted by TotalCount then Top1Count
    group_df = group_df.sort_values(['Top1Count', 'TotalCount'], ascending=[False, False]).reset_index(drop=True)
    display(group_df[['SuggestedName', 'OriginalSolver', 'Top1Count', 'TotalCount']].style.hide(axis="index"))


Suggestion counts — Variant: Features + Solvers Description / Temp: T0p3


SuggestedName,OriginalSolver,Top1Count,TotalCount
or-tools_cp-sat-free,atlantis-free,78,100
chuffed-free,highs-free,20,100
cp_optimizer-free,or-tools_cp-sat-free,2,45
picatsat-free,cp_optimizer-free,0,52
choco-solver__cp_-free,jacop-free,0,3



Suggestion counts — Variant: Features + Solvers Description / Temp: T0p7


SuggestedName,OriginalSolver,Top1Count,TotalCount
or-tools_cp-sat-free,atlantis-free,76,95
chuffed-free,highs-free,24,100
cp_optimizer-free,or-tools_cp-sat-free,0,74
picatsat-free,cp_optimizer-free,0,21
choco-solver__cp_-free,jacop-free,0,6
scip-free,choco-solver__cp-sat_-free,0,3
or-tools_cp-sat_ls-free,yuck-free,0,1



Suggestion counts — Variant: Scripts + Problem Description + Solvers Description / Temp: unknown


SuggestedName,OriginalSolver,Top1Count,TotalCount
or-tools_cp-sat-free,atlantis-free,56,95
chuffed-free,highs-free,40,100
atlantis-free,cplex-free,4,22
scip-free,choco-solver__cp-sat_-free,0,36
yuck-free,choco-solver__cp_-par,0,35
picatsat-free,cp_optimizer-free,0,6
cp_optimizer-free,or-tools_cp-sat-free,0,3
choco-solver__cp_-free,jacop-free,0,2
choco-solver__cp-sat_-free,sicstus_prolog-free,0,1



Suggestion counts — Variant: fzn2nl + Solver Description / Temp: T0p2


SuggestedName,OriginalSolver,Top1Count,TotalCount
or-tools_cp-sat-free,atlantis-free,72,99
atlantis-free,cplex-free,21,48
chuffed-free,highs-free,7,51
highs-free,or-tools_cp-sat_ls-free,0,78
yuck-free,choco-solver__cp_-par,0,24



Suggestion counts — Variant: fzn2nl + Solver Description / Temp: T0p7


SuggestedName,OriginalSolver,Top1Count,TotalCount
or-tools_cp-sat-free,atlantis-free,62,98
atlantis-free,cplex-free,32,53
chuffed-free,highs-free,6,49
highs-free,or-tools_cp-sat_ls-free,0,68
yuck-free,choco-solver__cp_-par,0,31
choco-solver__cp_-par,scip-free,0,1


In [65]:
# # --- Significative-only scoring pipeline (swapped) ---
# sig_solvers = get_significative_solvers()
# print(f"Using significative solvers (count={len(sig_solvers)}):")
# for s in sig_solvers:
#     print(" -", s)

# # MiniZinc results -> scores restricted to significative solvers
# mzn_raw_df = mznResultsFlattener(MznResults)
# scored_sig_df = scoreComputation_subset(mzn_raw_df, allowed_solvers=sig_solvers)
# print(f"\nMZN rows (raw): {len(mzn_raw_df)}")
# print(f"MZN rows (significative-only): {len(scored_sig_df)}")

# # Optional: show best single solver within significative set
# sig_single_solver_rank = singleSolverScore(scored_sig_df)
# display(sig_single_solver_rank.head(10))

# # LLM results -> compute Top-3, Top-1, ClosedGap within significative set
# tables_by_file = {}
# for fname, llm_results in data_by_file.items():
#     llm_df = ResultsFlattener(llm_results)
#     top3_summary = compute_llm_scores(llm_df, scored_sig_df, allowed_solvers=sig_solvers)
#     top1_summary, top1_scored = compute_top1_llm_scores(llm_df, scored_sig_df, allowed_solvers=sig_solvers)
#     cg_rows = compute_closed_gap(top1_scored, scored_sig_df, allowed_solvers=sig_solvers, sbs_solver=None)
#     cg_df = pd.DataFrame(cg_rows)
#     # Ensure `ClosedGap` column exists for downstream table building
#     if 'ClosedGap' not in cg_df.columns:
#         missing_cols = ['provider', 'model', 'InstancesCovered', 'AS', 'SBS', 'VBS', 'ClosedGap']
#         cg_df = cg_df.copy()
#         for c in missing_cols:
#             if c not in cg_df.columns:
#                 cg_df[c] = pd.NA
#     perf_table = build_llm_performance_table(
#         top3_summary=top3_summary,
#         top1_summary=top1_summary,
#         closed_gap=cg_df,
#         sort_by='SingleScore',
#         ascending=False,
#     )
#     tables_by_file[fname] = {
#         "top3_summary": top3_summary,
#         "top1_summary": top1_summary,
#         "closed_gap": cg_df,
#         "performance_table": perf_table,
#     }

# print(f"\nComputed significative-only scoring for {len(tables_by_file)} LLM result files.")

# # Display intermediate tables + performance table for each loaded file (stable order)
# for p in json_paths:
#     fname = p.name
#     if fname not in tables_by_file:
#         continue
#     print("\n" + "=" * 80)
#     print("File:", fname)
#     print("=" * 80)

#     print("\nTop-3 summary:")
#     display(tables_by_file[fname]["top3_summary"])

#     print("\nTop-1 summary:")
#     display(tables_by_file[fname]["top1_summary"])

#     print("\nClosed-gap table:")
#     display(tables_by_file[fname]["closed_gap"])

#     print("\nPerformance table:")
#     display(tables_by_file[fname]["performance_table"])

# # Compute and print the SBS/VBS that Closed Gap uses (for the current scoring subset).
# if "scored_sig_df" not in globals():
#     raise RuntimeError("Run the significative-only scoring pipeline above to create `scored_sig_df`.")

# # SBS: single solver with max total ComputedScore
# sbs_totals = scored_sig_df.groupby("Solver", as_index=False)["ComputedScore"].sum().rename(columns={"ComputedScore": "TotalComputedScore"})
# sbs_row = sbs_totals.sort_values("TotalComputedScore", ascending=False).head(1)
# SBS_SOLVER = str(sbs_row.iloc[0]["Solver"]) if not sbs_row.empty else ""
# SBS_TOTAL = float(sbs_row.iloc[0]["TotalComputedScore"]) if not sbs_row.empty else 0.0

# # VBS: per (Problem, Instance) max ComputedScore then sum
# vbs_df = (
#     scored_sig_df.groupby(["Problem", "Instance"], as_index=False)["ComputedScore"]
#     .max()
#     .rename(columns={"ComputedScore": "VBS_InstScore"})
# )
# VBS_TOTAL = float(vbs_df["VBS_InstScore"].sum()) if not vbs_df.empty else 0.0

# display(pd.DataFrame(
#     {
#         "Quantity": ["SBS Solver", "SBS Total", "VBS Total"],
#         "Value": [SBS_SOLVER, SBS_TOTAL, VBS_TOTAL],
#     }
# ))